<a href="https://colab.research.google.com/github/waltersalles/finguard-ai-consultant/blob/main/finguard_ai_consultant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
!pip install SpeechRecognition pydub gtts google-generativeai
!pip install SpeechRecognition pydub gtts google-generativeai -q
print("✅ Ferramentas instaladas!")

In [18]:
from IPython.display import HTML, display
from google.colab.output import eval_js
from base64 import b64decode

VIDEO_HTML = """
<script>
var my_recorder;
var my_stream;
async function startRecording() {
  my_stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  my_recorder = new MediaRecorder(my_stream);
  var chunks = [];
  my_recorder.ondataavailable = (e) => chunks.push(e.data);
  my_recorder.start();
  return new Promise(resolve => {
    my_recorder.onstop = async () => {
      var blob = new Blob(chunks);
      var reader = new FileReader();
      reader.readAsDataURL(blob);
      reader.onloadend = () => resolve(reader.result);
    };
  });
}
function stopRecording() {
  my_recorder.stop();
  my_stream.getTracks().forEach(i => i.stop());
}
</script>
<div style="background: #eef2f3; padding: 20px; border-radius: 15px; border: 1px solid #ccc; width: 300px; text-align: center;">
  <h3 style="color: #2c3e50;">FinGuard Voice</h3>
  <button onclick="startRecording()" style="background: #d9534f; color: white; padding: 10px; border: none; border-radius: 5px; cursor: pointer; margin: 5px;">🔴 Iniciar Consulta</button>
  <button onclick="stopRecording()" style="background: #5bc0de; color: white; padding: 10px; border: none; border-radius: 5px; cursor: pointer; margin: 5px;">⏹️ Finalizar Fala</button>
</div>
"""

def record_audio(filename='input_audio.wav'):
    display(HTML(VIDEO_HTML))
    data = eval_js("startRecording()")
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    print(f"✅ Áudio pronto para análise!")

record_audio()

✅ Áudio pronto para análise!


In [19]:
import google.generativeai as genai
from google.colab import userdata
import speech_recognition as sr
from pydub import AudioSegment
from gtts import gTTS
from IPython.display import Audio, display

# 1. Configuração do Cérebro
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))
instrucao = "Você é o FinGuard, consultor financeiro de 2026. Seja didático, mostre cálculos e use tom encorajador."
model = genai.GenerativeModel('gemini-2.5-flash', system_instruction=instrucao)

try:
    # 2. Tratamento do Áudio
    audio_original = AudioSegment.from_file("input_audio.wav")
    audio_original.export("input_audio_fixed.wav", format="wav")

    # 3. Escuta (STT)
    r = sr.Recognizer()
    with sr.AudioFile("input_audio_fixed.wav") as source:
        audio_data = r.record(source)
        pergunta = r.recognize_google(audio_data, language="pt-BR")
        print(f"👤 Você perguntou: {pergunta}")

    # 4. Inteligência (LLM)
    response = model.generate_content(pergunta)
    resposta_texto = response.text
    print(f"🤖 FinGuard: {resposta_texto}")

    # 5. Voz de Saída (TTS)
    tts = gTTS(text=f"Walter, analisei seu caso. {resposta_texto}", lang='pt')
    tts.save("resposta.mp3")
    display(Audio("resposta.mp3", autoplay=True))

except Exception as e:
    print(f"⚠️ Erro: {e}. Verifique se você gravou o áudio na Célula 2.")

⚠️ Erro: . Verifique se você gravou o áudio na Célula 2.
